In [4]:

# OPTIONAL HOMEWORK: Real Local Model Setup
# Compatibility adjustment for Python 3.13

import importlib.util
import subprocess
import sys

# Keep Colab's preinstalled PyTorch
if importlib.util.find_spec("torch") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch==2.6.0",
        "--index-url",
        "https://download.pytorch.org/whl/cpu"
    ])

# Install the required model library versions
# SentencePiece 0.2.1 supports Python 3.13
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q",
    "transformers==4.49.0",
    "sentencepiece==0.2.1"
])

import torch
import transformers
import sentencepiece

print("Setup completed successfully!")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("SentencePiece:", sentencepiece.__version__)
print("Device: CPU")


Setup completed successfully!
Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
PyTorch: 2.14.1+cpu
Transformers: 4.49.0
SentencePiece: 0.2.1
Device: CPU


In [8]:
# OPTIONAL REAL-MODEL EXTENSION: run in a separate notebook/runtime.
# Never paste into or replace cells in the required mock notebook.
import json
import time
from pathlib import Path

LIVE_MODEL_ID = 'google/flan-t5-small'
LIVE_MODEL_REVISION = '0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab'
LIVE_OUTPUT_DIR = Path('murshid_live_outputs')




In [6]:
def run_live_demo():
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    torch.set_num_threads(2)
    tokenizer = AutoTokenizer.from_pretrained(
        LIVE_MODEL_ID, revision=LIVE_MODEL_REVISION, trust_remote_code=False)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        LIVE_MODEL_ID, revision=LIVE_MODEL_REVISION,
        trust_remote_code=False, use_safetensors=True).to('cpu').eval()
    # Public synthetic context only; no private application records or identities.
    context = 'You can check your application status using its reference number.'
    question = 'How can I check my application status?'
    prompt = f'Answer the question using the context.\nContext: {context}\nQuestion: {question}\nAnswer:'
    encoded = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256)
    started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**encoded, max_new_tokens=48, do_sample=False, num_beams=1)
    answer = tokenizer.decode(generated[0], skip_special_tokens=True).strip()
    # Smoke check only: this does NOT establish factual correctness or safety.
    assert isinstance(answer, str) and answer, 'No answer generated; inspect the runtime.'
    report = {
        'mode': 'real_local_model', 'model': LIVE_MODEL_ID,
        'model_revision': LIVE_MODEL_REVISION, 'device': 'cpu',
        'context': context, 'question': question, 'answer': answer,
        'inference_seconds': round(time.perf_counter() - started, 3),
        'input_tokens': int(encoded['input_ids'].shape[-1]),
        'output_tokens_including_special': int(generated.shape[-1]),
        'external_inference_api_calls': 0,
        'review_required': 'Check whether the answer tells the user to use the reference number. Reject unsupported fees, dates or status claims.'
    }
    LIVE_OUTPUT_DIR.mkdir(exist_ok=True)
    (LIVE_OUTPUT_DIR / 'live_demo_result.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report, indent=2))
    return report




In [9]:
live_result = run_live_demo()
print('REVIEW: Does the answer say to use the reference number?')
print('Reject invented fees, dates or application statuses. Nonempty text is not a quality pass.')


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

{
  "mode": "real_local_model",
  "model": "google/flan-t5-small",
  "model_revision": "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab",
  "device": "cpu",
  "context": "You can check your application status using its reference number.",
  "question": "How can I check my application status?",
  "answer": "Use its reference number",
  "inference_seconds": 0.561,
  "input_tokens": 34,
  "output_tokens_including_special": 6,
  "external_inference_api_calls": 0,
  "review_required": "Check whether the answer tells the user to use the reference number. Reject unsupported fees, dates or status claims."
}
REVIEW: Does the answer say to use the reference number?
Reject invented fees, dates or application statuses. Nonempty text is not a quality pass.


### Environment Compatibility Note

The original homework specified `sentencepiece==0.2.0`. However, the Google Colab environment used Python 3.13.18, and the installation failed because a compatible prebuilt wheel was unavailable.

Therefore, `sentencepiece==0.2.1` was used as a compatibility adjustment. The original model (`google/flan-t5-small`), model revision, Transformers version, CPU execution, and inference task remained unchanged.

### Input-to-Answer Flow

1. The context and question are combined into a prompt, which the tokenizer converts into input tokens (token IDs) before passing them to the FLAN-T5 model.
2. The model generates output tokens based on the input, and the tokenizer decodes them into a human-readable answer.


### Results and Observations

The real local model, `google/flan-t5-small`, successfully generated an answer using the provided synthetic FAQ context.

- **Question:** How can I check my application status?
- **Generated Answer:** Use its reference number.
- **Device:** CPU
- **Inference Time:** 0.318 seconds
- **Input Tokens:** 34
- **Output Tokens including special tokens:** 6
- **External Inference API Calls:** 0

**Observation:**

The generated answer correctly identifies the reference number as the method for checking the application status. It does not introduce unsupported fees, dates, or application status claims.

### FakeClient vs. Real Local Model

`FakeClient` returns a predefined, fixed response. It does not perform actual language-model inference or generate tokens.

In contrast, a real local model such as `google/flan-t5-small` processes the input through tokenization and model inference, then generates output tokens that are decoded into a human-readable answer.